# Augmented Scene Generation

This notebook creates synthetic UNO scenes from the extracted reference-card crops.
Each scene contains 0 to 6 cards for each player, one center card, and one active-player token.
It saves RGB scenes under `training_images/augmented_scenes` and card segmentation masks under `training_masks/augmented_scenes`.

Masks are generated from per-card visible instances, then each instance is slightly eroded before the final binary mask is written. This creates a small black gap between overlapping cards so a segmenter can learn clean card boundaries.

## Path setup


In [ ]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd() / "project" / "notebooks" / "do" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate do/src directory for imports.")


## Imports


In [ ]:
from src.create_augmented_data import (
    CreateAugmentedDataConfig,
    initialize_create_augmented_data_pipeline,
    plot_card_preview,
    plot_saved_scenes,
    plot_scene_preview,
    run_card_generation,
    run_scene_generation,
    run_scene_preview,
)


## Configuration


In [ ]:
CFG = CreateAugmentedDataConfig()
CFG


## Initialize pipeline

Loads reference card crops, prepares output directories, and primes background/token caches.


In [ ]:
state = initialize_create_augmented_data_pipeline(CFG)
state.keys()


## Generate single-card augmentations

Creates `aug_*.jpg` crops for classifier training and the matching `aug.csv` labels file.


In [ ]:
state = run_card_generation(state)
len(state["aug_rows"])


In [ ]:
plot_card_preview(state)


## Preview a synthetic scene

Compose one scene in-memory before kicking off the full generation pass.


In [ ]:
state = run_scene_preview(state)
state["preview_metadata"]["style"], state["preview_metadata"]["active_player"]


In [ ]:
plot_scene_preview(state)


## Generate the augmented scene dataset


In [ ]:
state = run_scene_generation(state)
len(state["scene_metadata"])


## Visual check of saved scenes


In [ ]:
plot_saved_scenes(state)
